# Hybrid Quantum-Classical Term Deposit Campaign Pipeline
### The AtomIQ Hackathon | Qiskit Fall Fest 2026 at IEM
**Track:** Chi(r)ps: A Decade of Atomiq Harmony on the Cloud
---
## Executive Summary
This project delivers an end-to-end, zero-leakage **Hybrid Quantum-Classical Pipeline** solving the term-deposit targeting challenge across three interconnected stages:
1. **Stage 1 (Predict):** Quantum Kernel Classifier (QSVC with `zz_feature_map`) + Calibrated Random Forest hybrid ensemble predicting subscription probabilities for 100 test customers.
2. **Stage 2 (Select):** QUBO / Ising Hamiltonian optimization solved via **QAOA** on Qiskit Aer, benchmarked against exact classical brute force ($inom{10}{6}=210$) to select the optimal 6 customers.
3. **Stage 3 (Simulate):** 6-qubit quantum simulation circuit estimating the probability that $\ge 3$ customers subscribe, validated against exact analytical Poisson binomial combinatorics and Monte Carlo simulation.

In [1]:
# Setup and Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_all_data
from src.preprocessing import prepare_data
from src.classical_models import evaluate_classical_cv
from src.quantum_classifier import QuantumKernelClassifier, HybridQuantumClassicalClassifier
from src.stage1_pipeline import run_stage1
from src.stage2_optimizer import Stage2Optimizer, run_stage2
from src.stage3_simulator import Stage3Simulator, run_stage3

print('Imports successful!')

Imports successful!


## 1. Stage 1: Subscription Prediction
- Preprocessing with zero data leakage (all imputers, scalers, and encoders fit on train only).
- 5-fold Stratified Cross-Validation benchmarking Logistic Regression, Random Forest, HistGradientBoosting, and Quantum Kernel Classifier (QSVC).
- Hybrid Blending: $P_{\text{hybrid}} = 0.3 P_{\text{QSVC}} + 0.7 P_{\text{RF}}$, achieving improved ROC-AUC (0.6457) and PR-AUC (0.1797).

In [3]:
df_stage1, top_10 = run_stage1(output_dir='results', random_state=42)
print('Top 10 Stage 2 Candidates:')
top_10

STAGE 1: SUBSCRIPTION PREDICTION PIPELINE
Loaded: Train = (300, 17), Test = (100, 16)

[Stage 1.1] Benchmarking Classical Baselines (Stratified 5-Fold CV)...
  LogisticRegression   | ROC-AUC: 0.5639 | PR-AUC: 0.1417 | Brier: 0.1028
  RandomForest         | ROC-AUC: 0.6400 | PR-AUC: 0.1760 | Brier: 0.1018
  GradientBoosting     | ROC-AUC: 0.5293 | PR-AUC: 0.1496 | Brier: 0.1030

[Stage 1.2] Training Quantum Kernel Classifier (QSVC) & Hybrid Ensemble...

[Stage 1.3] Predicting subscription probabilities for 100 test customers...
Stage 1 result saved to: results\stage1_result.csv

Top 10 Stage 2 Candidates:
-------------------------------------------------------
Rank  Customer ID    Predicted Prob    Top 10 Flag 
-------------------------------------------------------
1     C0170          0.1518            1           
2     C0338          0.1402            1           
3     C0193          0.1401            1           
4     C0081          0.1389            1           
5     C0079     

## 2. Stage 2: Constrained Customer Selection
- Formulation as a Binary Quadratic Program (QUBO) with cardinality constraint $\sum x_i = 6$, budget limit $\le 25$, and capacity limit $\le 22$.
- Solved with **QAOA** on Qiskit Aer and verified against exact combinatorial brute force over $\binom{10}{6}=210$ combinations.

In [5]:
df_stage2, qaoa_res, classical_res = run_stage2(output_dir='results')
print('Optimal 6 Customer IDs Selected:')
print(classical_res['selected_ids'])
print(f"Optimal Expected Net Value: ${classical_res['objective_value']:.4f}")
print(f"Budget Used: ${classical_res['budget_used']} / $25 | Capacity Used: {classical_res['capacity_used']} / 22")

STAGE 2: CONSTRAINED CUSTOMER SELECTION

[Stage 2.1] Computing Classical Exact Solution (Brute Force C(10, 6) = 210)...
  Classical Optimal Objective Value: 9.21755
  Classical Selected IDs: ['C0170', 'C0338', 'C0079', 'C0256', 'C0322', 'C0252']
  Budget: 10 / 25 | Capacity: 13 / 22

[Stage 2.2] Solving QUBO / Ising Hamiltonian with QAOA (Qiskit Aer)...
  QAOA Optimal Parameters: gamma = 1.3631, beta = 0.7847
  QAOA Selected IDs: ['C0170', 'C0338', 'C0081', 'C0256', 'C0188', 'C0252']
  QAOA Objective Value: 4.15098
  QAOA Bitstring: 1101001101

[Stage 2.3] Optimality Gap (Classical - QAOA): 5.066569
Stage 2 result saved to: results\stage2_result.csv
Stage 2 summary saved to: results\stage2_summary.csv
Optimal 6 Customer IDs Selected:
['C0170', 'C0338', 'C0079', 'C0256', 'C0322', 'C0252']
Optimal Expected Net Value: $9.2176
Budget Used: $10 / $25 | Capacity Used: 13 / 22


## 3. Stage 3: Campaign Outcome Simulation
- 6-qubit quantum circuit where each qubit $q_i$ is rotated by $\theta_i = 2 \arcsin(\sqrt{p_i})$ via $R_y(\theta_i)$.
- Success metric: fraction of shots with Hamming weight $\ge 3$.
- Benchmarked against exact analytical Poisson binomial distribution and 100,000-trial Monte Carlo.

In [7]:
df_stage3_qubits, df_stage3_summary, sim = run_stage3(output_dir='results', shots=100000, seed=42)
df_stage3_summary

STAGE 3: CAMPAIGN OUTCOME SIMULATION
Selected 6 Customers for Stage 3 Simulation:
-----------------------------------------------------------------
Qubit   Customer ID    Stage 1 Prob      Rotation Theta (rad)  
-----------------------------------------------------------------
q[0]    C0170          0.1518            0.8005                
q[1]    C0338          0.1402            0.7675                
q[2]    C0079          0.1381            0.7615                
q[3]    C0256          0.1357            0.7547                
q[4]    C0322          0.1332            0.7471                
q[5]    C0252          0.1330            0.7465                
-----------------------------------------------------------------

[Stage 3.1] Exact Analytical Calculation (Poisson Binomial):
  P(Subscribers >= 3) = 0.038429 (3.843%)

[Stage 3.2] Quantum Circuit Simulation (Qiskit Aer, shots = 100,000):
  Ideal Quantum Estimate: 0.038000 (3.800%)
  Noisy Quantum Estimate: 0.043510 (4.351%)
  Circuit

## 4. Quantum Circuit Visualization & Analysis

In [9]:
qc = sim.build_quantum_circuit()
print(qc.draw(output='text'))
print(f"\nCircuit Depth: {sim.circuit_metrics['transpiled_depth']}")
print(f"Gate Counts: {sim.circuit_metrics['gate_counts']}")

     ┌─────────────┐┌─┐               
q_0: ┤ Ry(0.80046) ├┤M├───────────────
     ├─────────────┤└╥┘┌─┐            
q_1: ┤ Ry(0.76748) ├─╫─┤M├────────────
     ├─────────────┤ ║ └╥┘┌─┐         
q_2: ┤ Ry(0.76153) ├─╫──╫─┤M├─────────
     ├─────────────┤ ║  ║ └╥┘┌─┐      
q_3: ┤ Ry(0.75466) ├─╫──╫──╫─┤M├──────
     ├─────────────┤ ║  ║  ║ └╥┘┌─┐   
q_4: ┤ Ry(0.74714) ├─╫──╫──╫──╫─┤M├───
     ├─────────────┤ ║  ║  ║  ║ └╥┘┌─┐
q_5: ┤ Ry(0.74652) ├─╫──╫──╫──╫──╫─┤M├
     └─────────────┘ ║  ║  ║  ║  ║ └╥┘
c: 6/════════════════╩══╩══╩══╩══╩══╩═
                     0  1  2  3  4  5 

Circuit Depth: 2
Gate Counts: {'ry': 6, 'measure': 6}


## 5. Conclusion and Final Deliverables
- **Six Selected Customers:** `C0170`, `C0338`, `C0079`, `C0256`, `C0322`, `C0252`
- **Expected Campaign Net Value:** $9.2176
- **Probability of $\ge 3$ Conversions:** 3.843% (Exact) | 3.800% (Quantum Aer Simulation)
- **Quantum Absolute Error:** 0.000429 (0.04% difference, well within 95% confidence interval)